# Exercise XP — Air Traffic Data Analysis

**Inferential Statistics and Regression Analysis**

This notebook follows the 8 sections of the Developers Institute exercise. Place `air_traffic_data.csv` in the Colab Files panel before running it. If it is missing, a **synthetic demonstration dataset** is generated (not real air traffic data).

## 1. Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42
ALPHA = 0.05

In [ ]:
file_path = Path("air_traffic_data.csv")

if file_path.exists():
    df = pd.read_csv(file_path)
    print(f"Loaded real dataset: {file_path}")
else:
    print("WARNING: air_traffic_data.csv not found. Using SYNTHETIC sample data.")
    rng = np.random.default_rng(RANDOM_STATE)
    n = 240
    dom_flt = rng.integers(18_000, 60_000, n)
    int_flt = rng.integers(10_000, 38_000, n)
    dom_pax = (dom_flt * rng.normal(125, 12, n)).clip(min=1).round().astype(int)
    int_pax = (int_flt * rng.normal(155, 15, n)).clip(min=1).round().astype(int)
    df = pd.DataFrame({
        "Dom_Pax": dom_pax,
        "Int_Pax": int_pax,
        "Pax": dom_pax + int_pax,
        "Dom_Flt": dom_flt,
        "Int_Flt": int_flt,
        "Flt": dom_flt + int_flt,
        "Dom_RPM": (dom_pax * rng.normal(900, 100, n)).clip(min=1),
    })

required_columns = ["Dom_Pax", "Int_Pax", "Pax", "Dom_Flt", "Int_Flt", "Flt", "Dom_RPM"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

for col in required_columns:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False), errors="coerce")

print("Initial shape:", df.shape)
print("Missing values before cleaning:")
print(df[required_columns].isnull().sum())
df = df.dropna(subset=required_columns).copy()
print("Shape after dropping rows with missing required values:", df.shape)
if len(df) < 10:
    raise ValueError("Not enough complete observations to run the regression models.")
display(df.head())

## 2. Exploratory data analysis (EDA)

In [ ]:
df.info()
display(df.head())
display(df.describe())
print("Missing values by column:")
print(df.isnull().sum())

correlation_matrix = df[required_columns].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix — Air Traffic Data")
plt.tight_layout()
plt.show()

print("Correlations with total passengers (Pax):")
print(correlation_matrix["Pax"].drop("Pax").sort_values(ascending=False))
print("Note: a high correlation does not demonstrate causation.")

## 3. Hypothesis testing

In [ ]:
# Test 1 — Compare domestic and international passenger means
# H0: mean(Dom_Pax) = mean(Int_Pax)
# H1: mean(Dom_Pax) != mean(Int_Pax)
t_stat, p_value_t = stats.ttest_ind(df["Dom_Pax"], df["Int_Pax"], equal_var=False)
print("TEST 1 — Welch independent t-test")
print(f"Domestic mean: {df['Dom_Pax'].mean():,.2f}")
print(f"International mean: {df['Int_Pax'].mean():,.2f}")
print(f"t-statistic: {t_stat:.4f}; p-value: {p_value_t:.6g}")
print("Reject H0: means differ significantly." if p_value_t < ALPHA else "Fail to reject H0: no statistically significant difference detected.")
print("Caution: if domestic/international values are paired by period, a paired test may be more appropriate.")

# Test 2 — Pearson correlation between total passengers and total flights
# H0: population correlation = 0
# H1: population correlation != 0
r, p_value_r = stats.pearsonr(df["Pax"], df["Flt"])
print("\nTEST 2 — Pearson correlation")
print(f"r: {r:.4f}; p-value: {p_value_r:.6g}")
print("Reject H0: statistically significant correlation." if p_value_r < ALPHA else "Fail to reject H0: correlation not statistically significant.")
print("Direction:", "positive" if r > 0 else "negative" if r < 0 else "none")

## 4. Simple linear regression — predict `Pax` from `Flt`

In [ ]:
# Use the SAME train/test row split for both models for a fair comparison.
train_index, test_index = train_test_split(df.index, test_size=0.2, random_state=RANDOM_STATE)
y_train = df.loc[train_index, "Pax"]
y_test = df.loc[test_index, "Pax"]

X_simple_train = df.loc[train_index, ["Flt"]]
X_simple_test = df.loc[test_index, ["Flt"]]

simple_model = LinearRegression()
simple_model.fit(X_simple_train, y_train)
simple_predictions = simple_model.predict(X_simple_test)

def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "R2": r2_score(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_true, y_pred),
    }

simple_metrics = calculate_metrics(y_test, simple_predictions)
print(f"Equation: Pax = {simple_model.intercept_:.2f} + {simple_model.coef_[0]:.4f} × Flt")
print(pd.Series(simple_metrics).round(4))

simple_residuals = y_test - simple_predictions
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_test, y=simple_predictions)
low = min(y_test.min(), simple_predictions.min())
high = max(y_test.max(), simple_predictions.max())
plt.plot([low, high], [low, high], "r--", label="Perfect predictions")
plt.xlabel("Actual Pax")
plt.ylabel("Predicted Pax")
plt.title("Simple Regression — Actual vs Predicted")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(x=simple_predictions, y=simple_residuals)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Pax")
plt.ylabel("Residuals")
plt.title("Simple Regression — Residual Plot")
plt.tight_layout()
plt.show()

## 5. Multiple linear regression

The assignment specifically suggests `Dom_Pax`, `Int_Pax`, `Dom_Flt`, `Int_Flt`, and `Dom_RPM`. **Important:** `Pax = Dom_Pax + Int_Pax`. Including both passenger-count components as features reveals the target mathematically (*target leakage*), so the model's test results will be artificially perfect. The notebook keeps those predictors to match the assignment and explicitly flags that limitation.

In [ ]:
multiple_features = ["Dom_Pax", "Int_Pax", "Dom_Flt", "Int_Flt", "Dom_RPM"]
X_multiple_train = df.loc[train_index, multiple_features]
X_multiple_test = df.loc[test_index, multiple_features]

# Fit the scaler ONLY on the training data.
scaler = StandardScaler()
X_multiple_train_scaled = scaler.fit_transform(X_multiple_train)
X_multiple_test_scaled = scaler.transform(X_multiple_test)

multiple_model = LinearRegression()
multiple_model.fit(X_multiple_train_scaled, y_train)
multiple_predictions = multiple_model.predict(X_multiple_test_scaled)
multiple_metrics = calculate_metrics(y_test, multiple_predictions)

print("Multiple regression metrics:")
print(pd.Series(multiple_metrics).round(4))
print("Coefficients (in standardized-feature units):")
display(pd.Series(multiple_model.coef_, index=multiple_features, name="Coefficient").to_frame())

multiple_residuals = y_test - multiple_predictions
plt.figure(figsize=(8, 5))
sns.scatterplot(x=multiple_predictions, y=multiple_residuals)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Pax")
plt.ylabel("Residuals")
plt.title("Multiple Regression — Residual Plot")
plt.tight_layout()
plt.show()
print("WARNING: Dom_Pax + Int_Pax equals Pax, creating target leakage.")

## 6. Model comparison and analysis

In [ ]:
comparison = pd.DataFrame({
    "Simple Regression": simple_metrics,
    "Multiple Regression": multiple_metrics,
}).T

display(comparison.round(4))

r2_change = multiple_metrics["R2"] - simple_metrics["R2"]
rmse_improvement_pct = (simple_metrics["RMSE"] - multiple_metrics["RMSE"]) / simple_metrics["RMSE"] * 100 if simple_metrics["RMSE"] else np.nan
mae_improvement_pct = (simple_metrics["MAE"] - multiple_metrics["MAE"]) / simple_metrics["MAE"] * 100 if simple_metrics["MAE"] else np.nan

print(f"R² change (multiple - simple): {r2_change:+.4f} points")
print(f"RMSE reduction: {rmse_improvement_pct:.2f}%")
print(f"MAE reduction: {mae_improvement_pct:.2f}%")
best_model = comparison["RMSE"].idxmin()
print(f"Lower test RMSE: {best_model}")
print("However, the multiple model uses components of the target, so this is NOT a fair real-world predictive comparison.")

## 7. Statistical insights and conclusions

In [ ]:
print("=== DATA-DRIVEN SUMMARY ===")
print("Domestic vs international mean passengers:",
      "significant difference" if p_value_t < ALPHA else "no significant difference detected",
      f"(p={p_value_t:.4g}).")
print(f"Passenger–flight correlation: r={r:.3f}, p={p_value_r:.4g}.")
print(f"Simple regression: R²={simple_metrics['R2']:.3f}, RMSE={simple_metrics['RMSE']:,.0f}, MAE={simple_metrics['MAE']:,.0f}.")
print(f"Multiple regression: R²={multiple_metrics['R2']:.3f}, RMSE={multiple_metrics['RMSE']:,.0f}, MAE={multiple_metrics['MAE']:,.0f}.")
print("Business application: flight counts may support passenger-volume forecasting and capacity planning.")
print("Do not infer causation from correlation. Validate on future time periods before operational deployment.")
print("Critical limitation: multiple-model predictions are invalid for advance forecasting because they use actual passenger counts.")

## 8. Reflection questions

1. **What do hypothesis tests reveal?** The first test examines a difference in average domestic vs international passenger counts; the second examines the strength and significance of the relationship between passenger volume and flights. Interpret the actual p-values printed above.
2. **Why did one regression perform better?** In this prescribed exercise, the multiple model benefits from `Dom_Pax + Int_Pax = Pax`, which is leakage, not a genuine predictive improvement.
3. **How can airlines use correlations?** For demand and capacity planning, while checking seasonality, delays, economic conditions and additional predictors.
4. **What do residual plots show?** Systematic curves or widening spread suggest possible violations of linearity or constant-variance assumptions; random scatter around zero is more reassuring.
5. **Practical applications?** Forecasting flight demand, staffing, gate usage and scheduling after removing leakage and validating on future unseen data.

**Optional improvement:** For genuine forecasting, retrain the multiple model *without* `Dom_Pax` and `Int_Pax` and use predictors observable before the predicted period.